# Modelo y validación

Entrena un Random Forest de clasificación que predice el nivel de ocupación (0 a 3)
a partir de las características de cada ventana, y mide lo bien que acierta en datos
que no ha visto.

Diapositivas: clase 7, 63 (aprendizaje supervisado), 65 (clasificación),
67–69 (Random Forest) y 53–54 (muestreo estratificado).

**Aviso:** con los datos simulados los resultados saldrán muy buenos porque las
relaciones entre sensores y ocupación son inventadas. Este notebook sirve para tener
el proceso listo; los resultados de verdad saldrán con datos reales.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("No se encuentra config.yaml: abre el notebook dentro del repositorio")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from occupancy.cleaning import clean
from occupancy.integration import merge_all
from occupancy.features import build_windows

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
with open(ROOT / "data" / "simulated" / "readings.json", encoding="utf-8") as f:
    data = json.load(f)["data"]

# Fases 1, 2 y 4
tables, pir, labels = clean(data, config["cleaning"])
df = merge_all(tables, pir, labels)
windows = build_windows(df, list(tables), config["windows"], config["cleaning"]["interval_seconds"])

print("Ventanas:", windows.shape)
print(windows["label"].value_counts().sort_index())

## Paso 1: separar características (X) y respuesta (y)

In [ ]:
X = windows.drop(columns="label")   # características de cada ventana
y = windows["label"]                # nivel real (0 a 3), lo que el modelo debe aprender

print("X:", X.shape, " y:", y.shape)

## Paso 2: grupos para la validación

Ventanas seguidas se parecen mucho. Si unas van a entrenar y sus vecinas a probar, el
resultado sale demasiado bueno. Por eso se agrupan en bloques de 15 minutos y cada
bloque entero va a entrenar o a probar. Con datos reales, el grupo será la sesión.

In [ ]:
grupos = windows.index.floor("15min").astype(str)   # bloque de 15 min de cada ventana

print("Bloques:", grupos.nunique())

## Paso 3: entrenar y probar en cada partición

Se divide en 4 partes manteniendo la proporción de niveles (muestreo estratificado,
diapositivas 53–54). Cada vez se entrena con 3 partes y se prueba con la que queda, así
cada ventana se predice con un modelo que no la ha visto. Los huecos se rellenan con la
mediana de los datos de entrenamiento.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold

particiones = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
reales, predichos = [], []

for entreno, prueba in particiones.split(X, y, grupos):
    relleno = X.iloc[entreno].median()
    modelo = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    modelo.fit(X.iloc[entreno].fillna(relleno), y.iloc[entreno])

    reales.extend(y.iloc[prueba])
    predichos.extend(modelo.predict(X.iloc[prueba].fillna(relleno)))

reales, predichos = np.array(reales), np.array(predichos)

## Paso 4: métricas

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

niveles = [0, 1, 2, 3]
print("Exactitud:", round(accuracy_score(reales, predichos), 3))
print("Acierto con margen de un nivel:", round(np.mean(np.abs(reales - predichos) <= 1), 3))
print()
print(pd.DataFrame(confusion_matrix(reales, predichos, labels=niveles),
                   index=[f"real {n}" for n in niveles],
                   columns=[f"pred {n}" for n in niveles]))
print()
print(classification_report(reales, predichos, labels=niveles, zero_division=0))

## Paso 5: entrenar el modelo final y ver qué características usa más

El modelo final se entrena con todas las ventanas. La importancia dice cuánto usa cada
característica (suman 1); las que salen casi a cero son candidatas a quitarse
(reducción, diapositivas 34–46).

In [ ]:
relleno = X.median()
modelo_final = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
modelo_final.fit(X.fillna(relleno), y)

pd.Series(modelo_final.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

## Paso 6: guardar el modelo final

Se guarda con lo que `predict.py` necesita para usarlo en directo: el orden de las
columnas, los valores de relleno y el rango visto al entrenar.

In [ ]:
modelo_guardado = {
    "model": modelo_final,
    "features": list(X.columns),
    "fill_values": relleno,
    "feature_min": X.min(),
    "feature_max": X.max(),
}
ruta = ROOT / config["model"]["path"]
ruta.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_guardado, ruta)
print("Modelo guardado en", ruta)